In [2]:
!pip install ucimlrepo

In [8]:
# Graph-Based Clustering using MST (Final Colab Version)

# 1. Import Libraries
import numpy as np
import pandas as pd
from scipy.spatial.distance import pdist, squareform
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# 2. Load Dataset (Iris)
from sklearn.datasets import load_iris
data = load_iris()
df = pd.DataFrame(data.data, columns=data.feature_names)

X = df.values
print("Dataset Loaded. Shape:", X.shape)

# 3. MST Clustering Function
def mst_clustering(X, k=3, metric='euclidean'):
    dist_matrix = squareform(pdist(X, metric=metric))
    mst = minimum_spanning_tree(dist_matrix).toarray()

    # Extract edges
    edges = [(i, j, mst[i, j]) for i in range(len(X))
             for j in range(len(X)) if mst[i, j] > 0]

    # Sort edges (descending)
    edges = sorted(edges, key=lambda x: x[2], reverse=True)

    # Remove (k-1) largest edges
    for i in range(k - 1):
        u, v, _ = edges[i]
        mst[u, v] = 0

    # DFS for cluster labeling
    visited = [False] * len(X)
    labels = [-1] * len(X)
    cluster_id = 0

    def dfs(node):
        stack = [node]
        while stack:
            curr = stack.pop()
            if not visited[curr]:
                visited[curr] = True
                labels[curr] = cluster_id
                for neighbor in range(len(X)):
                    if mst[curr, neighbor] > 0 or mst[neighbor, curr] > 0:
                        if not visited[neighbor]:
                            stack.append(neighbor)

    for i in range(len(X)):
        if not visited[i]:
            dfs(i)
            cluster_id += 1

    return labels


# 4. PARAMETERS
k = 3


# WITHOUT SCALING

print("\n--- WITHOUT SCALING ---")

mst_raw = mst_clustering(X, k=k)
kmeans_raw = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)

mst_score_raw = silhouette_score(X, mst_raw)
kmeans_score_raw = silhouette_score(X, kmeans_raw)

print("MST:", mst_score_raw)
print("K-Means:", kmeans_score_raw)

# Save outputs
pd.DataFrame({
    "SampleId": range(len(mst_raw)),
    "ClusterLabel": mst_raw
}).to_csv("mst_clusters_raw.csv", index=False)

pd.DataFrame({
    "SampleId": range(len(kmeans_raw)),
    "ClusterLabel": kmeans_raw
}).to_csv("kmeans_clusters_raw.csv", index=False)



# WITH SCALING

print("\n--- WITH SCALING ---")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

mst_scaled = mst_clustering(X_scaled, k=k)
kmeans_scaled = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)

mst_score_scaled = silhouette_score(X_scaled, mst_scaled)
kmeans_score_scaled = silhouette_score(X_scaled, kmeans_scaled)

print("MST:", mst_score_scaled)
print("K-Means:", kmeans_score_scaled)

# Save outputs
pd.DataFrame({
    "SampleId": range(len(mst_scaled)),
    "ClusterLabel": mst_scaled
}).to_csv("mst_clusters_scaled.csv", index=False)

pd.DataFrame({
    "SampleId": range(len(kmeans_scaled)),
    "ClusterLabel": kmeans_scaled
}).to_csv("kmeans_clusters_scaled.csv", index=False)

print("\nCluster files saved!")


# 📊 FINAL COMPARISON REPORT

print("\n\t--- FINAL COMPARISON REPORT ---")

print(f"""
WITHOUT SCALING:
MST Score      : {mst_score_raw:.4f}
K-Means Score  : {kmeans_score_raw:.4f}

WITH SCALING:
MST Score      : {mst_score_scaled:.4f}
K-Means Score  : {kmeans_score_scaled:.4f}
""")

print("""
Observations:
- K-Means performs better without scaling because certain features (like petal length)
  dominate and help separate clusters clearly.
- After scaling, all features contribute equally, reducing natural separation,
  which lowers K-Means performance.
- MST clustering is more stable because it depends on relative distances
  and local structure rather than absolute magnitudes.

Conclusion:
- K-Means is best for compact, well-separated clusters.
- MST is useful for detecting irregular and non-spherical clusters
  but is sensitive to chaining effects.
""")


# EXPLANATION

print("\n\t--- EXPLANATION ---")
print("""
Minimum Spanning Tree (MST) clustering connects all data points with the minimum total edge weight using Prim’s algorithm.
By removing the (k−1) largest edges, weak connections between dense regions are eliminated to form clusters.
This approach does not assume any specific cluster shape, allowing detection of non-spherical and irregular clusters.
Unlike K-Means, which assumes spherical clusters, MST can capture complex structures in data.
However, MST is sensitive to noise and may suffer from the chaining effect, where clusters are incorrectly linked.
K-Means performs better when clusters are compact and well-separated, as seen in the Iris dataset.
""")

Dataset Loaded. Shape: (150, 4)

--- WITHOUT SCALING ---
MST: 0.5121107753649307
K-Means: 0.5528190123564095

--- WITH SCALING ---
MST: 0.504645610832545
K-Means: 0.45994823920518635

Cluster files saved!

	--- FINAL COMPARISON REPORT ---

WITHOUT SCALING:
MST Score      : 0.5121
K-Means Score  : 0.5528

WITH SCALING:
MST Score      : 0.5046
K-Means Score  : 0.4599


Observations:
- K-Means performs better without scaling because certain features (like petal length)
  dominate and help separate clusters clearly.
- After scaling, all features contribute equally, reducing natural separation,
  which lowers K-Means performance.
- MST clustering is more stable because it depends on relative distances
  and local structure rather than absolute magnitudes.

Conclusion:
- K-Means is best for compact, well-separated clusters.
- MST is useful for detecting irregular and non-spherical clusters
  but is sensitive to chaining effects.


	--- EXPLANATION ---

Minimum Spanning Tree (MST) clustering 